# Ellipsis split analysis

Inspect locally exported transcription segments containing `...`, preview a first-ellipsis split, and listen to both sides using local MP3 files.

This notebook is read-only: it does not modify exported transcripts.

In [6]:
import json
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

RUN_ID = "4bd7a1a8d9024fd08c00c0e238bcc6b8"
ELLIPSIS_RE = re.compile(r"\.{3,}")
PAD_SECONDS = 0.25

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent

run_root = repo / "data" / "mlflow_exports" / RUN_ID
audio_root = repo / "downloads"
config = json.loads((run_root / "config.json").read_text(encoding="utf-8"))

if not (run_root / "episodes").is_dir():
    raise FileNotFoundError(f"Missing local export: {run_root}")
if not audio_root.is_dir():
    raise FileNotFoundError(f"Missing local audio: {audio_root}")

In [7]:
def episode_sort_key(path: Path):
    key = path.parents[1].name
    return (0, int(key)) if key.isdigit() else (1, key)


def split_preview(segment: dict) -> tuple[str, str, float]:
    text = segment.get("text", "")
    match = ELLIPSIS_RE.search(text)
    if match is None:
        raise ValueError("Segment has no ellipsis")

    left = text[: match.start()].strip()
    right = text[match.end() :].lstrip(" .")
    words = segment.get("words", [])
    split_at = (segment["start"] + segment["end"]) / 2

    for index, word in enumerate(words):
        token = word.get("word", "")
        if "..." not in token:
            continue
        if token.rstrip().endswith("...") and index + 1 < len(words):
            split_at = (word["end"] + words[index + 1]["start"]) / 2
        elif token.startswith("...") and index > 0:
            split_at = (words[index - 1]["end"] + word["start"]) / 2
        else:
            split_at = (word.get("start", split_at) + word.get("end", split_at)) / 2
        break

    return left, right, split_at


def load_candidates() -> tuple[pd.DataFrame, int, int]:
    rows = []
    transcript_count = 0
    segment_count = 0
    paths = sorted(
        (run_root / "episodes").glob("*/transcription/transcript.json"),
        key=episode_sort_key,
    )
    for path in paths:
        episode_key = path.parents[1].name
        try:
            segments = json.loads(path.read_text(encoding="utf-8"))["segments"]
        except json.JSONDecodeError:
            continue

        assignment_path = path.parents[1] / "character_assigner" / "transcript.json"
        try:
            assigned = json.loads(
                assignment_path.read_text(encoding="utf-8")
            )["segments"]
        except (FileNotFoundError, json.JSONDecodeError):
            assigned = []

        transcript_count += 1
        segment_count += len(segments)

        for index, segment in enumerate(segments):
            text = segment.get("text", "")
            if "..." not in text:
                continue
            left, right, split_at = split_preview(segment)
            rows.append(
                {
                    "episode": episode_key,
                    "segment_index": index,
                    "start": segment["start"],
                    "split_at": split_at,
                    "end": segment["end"],
                    "assigned_speaker": (
                        assigned[index].get("speaker") if index < len(assigned) else None
                    ),
                    "ellipsis_count": len(ELLIPSIS_RE.findall(text)),
                    "double_ellipsis": "... ..." in text,
                    "valid_split": bool(left and right),
                    "text": text,
                    "left": left,
                    "right": right,
                }
            )

    return pd.DataFrame(rows), transcript_count, segment_count


candidates, transcript_count, segment_count = load_candidates()
display(
    pd.Series(
        {
            "transcripts scanned": transcript_count,
            "segments scanned": segment_count,
            "segments containing ...": len(candidates),
            "share of segments": len(candidates) / segment_count if segment_count else 0,
            "segments splittable at first ...": int(candidates["valid_split"].sum()),
            "segments containing ... ...": int(candidates["double_ellipsis"].sum()),
            "segments with multiple ellipses": int((candidates["ellipsis_count"] > 1).sum()),
        },
        name="value",
    ).to_frame()
)

,value
transcripts scanned,597.000000
segments scanned,90873.000000
segments containing ...,4825.000000
share of segments,0.053096
segments splittable at first ...,3644.000000
segments containing ... ...,7.000000
segments with multiple ellipses,529.000000


In [ ]:
# Narrow the review queue without changing the underlying candidate table.
VALID_SPLITS_ONLY = True
ONLY_DOUBLE_ELLIPSIS = False

review = candidates
if VALID_SPLITS_ONLY:
    review = review[review["valid_split"]]
if ONLY_DOUBLE_ELLIPSIS:
    review = review[review["double_ellipsis"]]
review = review.reset_index(drop=True)

pd.set_option("display.max_colwidth", 160)
display(
    review[
        [
            "episode",
            "segment_index",
            "start",
            "split_at",
            "end",
            "assigned_speaker",
            "ellipsis_count",
            "double_ellipsis",
            "text",
            "left",
            "right",
        ]
    ]
)

In [8]:
import ipywidgets as widgets
from IPython.display import Audio, clear_output

from drzavni_posao import MediaStore
from drzavni_posao.constants import SR

media = MediaStore(
    audio_root,
    run_root,
    trim_start=config["audio"]["trim_start"],
    trim_end=config["audio"]["trim_end"],
)

position = 0
output = widgets.Output()
previous_button = widgets.Button(description="Previous")
replay_button = widgets.Button(description="Replay")
next_button = widgets.Button(description="Next")
buttons = [previous_button, replay_button, next_button]


def show_candidate():
    with output:
        clear_output(wait=True)
        if review.empty:
            print("No matching segments.")
            for button in buttons:
                button.disabled = True
            return

        row = review.iloc[position]
        print(
            f"{position + 1}/{len(review)}  episode {row.episode}  "
            f"segment {row.segment_index}  "
            f"{row.start:.3f}–{row.end:.3f}s"
        )
        print(f"current assignment: {row.assigned_speaker or '[none]'}")
        print(f"LEFT:  {row.left}")
        print(f"RIGHT: {row.right}")

        try:
            episode = media.episode(str(row.episode))
        except KeyError as error:
            print(error)
            return

        waveform = episode.audio()
        clip_start = max(0, int((row.start - PAD_SECONDS) * SR))
        split_at = int(row.split_at * SR)
        clip_end = min(len(waveform), int((row.end + PAD_SECONDS) * SR))
        full_clip = waveform[clip_start:clip_end].copy()
        left_clip = waveform[clip_start:min(clip_end, split_at)].copy()
        right_clip = waveform[max(clip_start, split_at):clip_end].copy()
        episode.clear_audio()

        print("FULL")
        display(Audio(full_clip, rate=SR, autoplay=True))
        print("PROPOSED LEFT / RIGHT")
        display(Audio(left_clip, rate=SR), Audio(right_clip, rate=SR))


def step(delta: int):
    global position
    position = max(0, min(len(review) - 1, position + delta))
    show_candidate()


previous_button.on_click(lambda _: step(-1))
replay_button.on_click(lambda _: show_candidate())
next_button.on_click(lambda _: step(1))
display(widgets.VBox([output, widgets.HBox(buttons)]))
show_candidate()

In [9]:
# Review segments with both weak ASR probability and forced-alignment confidence.
LOW_AVG_LOGPROB = -0.5
LOW_MEAN_WORD_SCORE = 0.5
REQUIRE_BOTH = True


def load_low_confidence_segments() -> pd.DataFrame:
    rows = []
    paths = sorted(
        (run_root / "episodes").glob("*/transcription/transcript.json"),
        key=episode_sort_key,
    )
    for path in paths:
        episode_key = path.parents[1].name
        try:
            segments = json.loads(path.read_text(encoding="utf-8"))["segments"]
        except json.JSONDecodeError:
            continue

        assignment_path = path.parents[1] / "character_assigner" / "transcript.json"
        try:
            assigned = json.loads(
                assignment_path.read_text(encoding="utf-8")
            )["segments"]
        except (FileNotFoundError, json.JSONDecodeError):
            assigned = []

        for index, segment in enumerate(segments):
            avg_logprob = segment.get("avg_logprob")
            word_scores = [
                word["score"]
                for word in segment.get("words", [])
                if word.get("score") is not None
            ]
            mean_word_score = (
                sum(word_scores) / len(word_scores) if word_scores else None
            )
            low_probability = (
                avg_logprob is not None and avg_logprob <= LOW_AVG_LOGPROB
            )
            low_alignment = (
                mean_word_score is not None
                and mean_word_score <= LOW_MEAN_WORD_SCORE
            )
            selected = (
                low_probability and low_alignment
                if REQUIRE_BOTH
                else low_probability or low_alignment
            )
            if not selected:
                continue

            reasons = []
            if low_probability:
                reasons.append("ASR probability")
            if low_alignment:
                reasons.append("alignment confidence")
            rows.append(
                {
                    "episode": episode_key,
                    "segment_index": index,
                    "start": segment["start"],
                    "end": segment["end"],
                    "duration": segment["end"] - segment["start"],
                    "avg_logprob": avg_logprob,
                    "mean_word_score": mean_word_score,
                    "reason": " + ".join(reasons),
                    "assigned_speaker": (
                        assigned[index].get("speaker") if index < len(assigned) else None
                    ),
                    "text": segment.get("text", ""),
                }
            )

    if not rows:
        return pd.DataFrame(rows)
    return pd.DataFrame(rows).sort_values(
        ["avg_logprob", "mean_word_score"],
        na_position="last",
    ).reset_index(drop=True)


low_confidence = load_low_confidence_segments()
operator = "and" if REQUIRE_BOTH else "or"
print(
    f"{len(low_confidence)} segments matched "
    f"(avg_logprob <= {LOW_AVG_LOGPROB} {operator} "
    f"mean word score <= {LOW_MEAN_WORD_SCORE})"
)
display(low_confidence.head(200))

low_position = 0
low_output = widgets.Output()
low_previous = widgets.Button(description="Previous")
low_replay = widgets.Button(description="Replay")
low_next = widgets.Button(description="Next")
low_buttons = [low_previous, low_replay, low_next]


def show_low_confidence():
    with low_output:
        clear_output(wait=True)
        if low_confidence.empty:
            print("No low-confidence segments at these thresholds.")
            for button in low_buttons:
                button.disabled = True
            return

        row = low_confidence.iloc[low_position]
        print(
            f"{low_position + 1}/{len(low_confidence)}  episode {row.episode}  "
            f"segment {row.segment_index}  {row.start:.3f}–{row.end:.3f}s"
        )
        print(
            f"avg_logprob={row.avg_logprob:.3f}  "
            f"mean_word_score={row.mean_word_score:.3f}  reason={row.reason}"
        )
        print(f"current assignment: {row.assigned_speaker or '[none]'}")
        print(row.text)

        try:
            episode = media.episode(str(row.episode))
        except KeyError as error:
            print(error)
            return

        waveform = episode.audio()
        clip_start = max(0, int((row.start - PAD_SECONDS) * SR))
        clip_end = min(len(waveform), int((row.end + PAD_SECONDS) * SR))
        clip = waveform[clip_start:clip_end].copy()
        episode.clear_audio()
        display(Audio(clip, rate=SR, autoplay=True))


def step_low_confidence(delta: int):
    global low_position
    low_position = max(
        0,
        min(len(low_confidence) - 1, low_position + delta),
    )
    show_low_confidence()


low_previous.on_click(lambda _: step_low_confidence(-1))
low_replay.on_click(lambda _: show_low_confidence())
low_next.on_click(lambda _: step_low_confidence(1))
display(widgets.VBox([low_output, widgets.HBox(low_buttons)]))
show_low_confidence()

209 segments matched (avg_logprob <= -0.5 and mean word score <= 0.5)


,episode,segment_index,start,end,duration,avg_logprob,mean_word_score,reason,assigned_speaker,text
0,536,131,442.810,442.951,0.141,-2.515625,0.414000,ASR probability + alignment confidence,Djordje Cvarkov,...
1,129,134,345.206,345.266,0.060,-2.031250,0.000000,ASR probability + alignment confidence,[anonymous],...
2,457,35,137.039,137.119,0.080,-2.015625,0.158000,ASR probability + alignment confidence,[anonymous],...
3,8,64,161.538,161.598,0.060,-1.976562,0.000000,ASR probability + alignment confidence,[anonymous],...
4,8,63,152.035,153.017,0.982,-1.916667,0.460000,ASR probability + alignment confidence,Dragan Torbica,... ...
...,...,...,...,...,...,...,...,...,...,...
195,"DRŽAVNI POSAO [HQ] - Novogodišnji specijal, 1.deo",1,6.195,6.476,0.281,-0.513889,0.159000,ASR probability + alignment confidence,[anonymous],pročutio.
196,379,17,73.847,74.187,0.340,-0.513393,0.422500,ASR probability + alignment confidence,Dragan Torbica,Kako ste?
197,348,1,3.837,6.375,2.538,-0.512821,0.219125,ASR probability + alignment confidence,Boskic,"Kaže, evo Milka Planinac, kaže, vaši tani, smo"
198,385,124,352.374,353.004,0.630,-0.510965,0.378333,ASR probability + alignment confidence,Djordje Cvarkov,"Dodaj mi pivu,"
